# 4.1 Feature Selection Optimizada para Predicción Direccional

Este notebook implementa un pipeline de feature selection **desde cero**, partiendo del dataset completo con ~3200 features (`features_final_modeling.csv`).

## Objetivos

1. **Cargar dataset completo** (3236 features de notebooks 2.x)
2. **Definir targets de clasificación** para horizontes: t+1, t+5, t+21 días
3. **Aplicar pipeline de feature selection** optimizado
4. **Generar subsets de features óptimos** por commodity y horizonte

## Horizontes Temporales (estándar industria)
- **t+1**: 1 día de trading
- **t+5**: ~1 semana (5 días hábiles)
- **t+21**: ~1 mes (21 días hábiles)

In [54]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json
from datetime import datetime
import matplotlib.pyplot as plt
import gc

from sklearn.feature_selection import mutual_info_classif, VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Path setup
BASE_DIR = Path.cwd().parents[1]
sys.path.append(str(BASE_DIR / 'src'))
from config import PROCESSED_DIR

# Configuración
TARGET_COMMODITIES = ['Corn', 'Soybeans', 'Wheat']
HORIZONS = {'t1': 1, 't5': 5, 't21': 21}
RANDOM_STATE = 42

print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ Commodities: {TARGET_COMMODITIES}")
print(f"✓ Horizontes: {HORIZONS}")

✓ Base directory: c:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal
✓ Commodities: ['Corn', 'Soybeans', 'Wheat']
✓ Horizontes: {'t1': 1, 't5': 5, 't21': 21}


In [55]:
# =============================================================================
# CORRECCIONES BASADAS EN VALIDACIÓN CON LITERATURA ACADÉMICA
# =============================================================================
# Referencia: PLAN_CORRECCION_FEATURE_SELECTION.md

# 1. WHITELIST: Features que DEBEN incluirse (validadas por literatura)
# Sentiment features (Tetlock 2007, Manela & Moreira 2017)
WHITELIST_FEATURES = [
    'tone_mean', 'tone_std', 'tone_ma7', 'tone_ma30',
    'tone_volatility_7d', 'tone_volatility_30d', 
    'tone_percentile_30d', 'tone_momentum_7d',
    'article_count', 'article_count_change'
]

# 2. BLACKLIST: Patrones a excluir (correlaciones espurias)
# Volumen de metales preciosos NO predice granos (sin mecanismo causal)
BLACKLIST_PATTERNS = {
    'Corn': ['Gold_volume', 'Silver_volume', 'Platinum_volume', 'Palladium_volume'],
    'Soybeans': ['Gold_volume', 'Silver_volume', 'Platinum_volume', 'Palladium_volume'],
    'Wheat': ['Gold_volume', 'Silver_volume', 'Platinum_volume', 'Palladium_volume']
}

# 3. DATA LEAKAGE: Patrones a excluir POR HORIZONTE
# CFTC COT data: publicación viernes con 3 días de delay (Tuesday positions)
# Government stocks: publicación mensual con varios días de delay
HORIZON_BLACKLIST = {
    't1': [
        # CFTC COT - 3+ días de delay de publicación
        'other_long', 'other_short', 'other_net', 'other_spreading',
        'managed_long', 'managed_short', 'managed_net', 'managed_spreading',
        'producer_long', 'producer_short', 'producer_net', 'producer_spreading',
        'swap_long', 'swap_short', 'swap_net', 'swap_spreading',
        'dealer_long', 'dealer_short', 'dealer_net', 'dealer_spreading',
        'nonreportable_long', 'nonreportable_short', 'nonreportable_net',
        # Government stocks - monthly data with delay
        'gov_stocks', 'gov_stocks_change', 'gov_stocks_pct_change',
        # WASDE/PSD - monthly releases
        'psd_', 'wasde_', 'usda_'
    ],
    't5': [],  # 5 días > 3 días delay, OK para CFTC
    't21': []  # 21 días >> cualquier delay, OK
}

# 3. LÍMITE DE FEATURES is_outlier (evitar overfitting)
# Mantener solo top 10 por MI score
MAX_OUTLIER_FEATURES = 10

print("✓ Configuración de correcciones cargada:")
print(f"  - Whitelist: {len(WHITELIST_FEATURES)} features de sentiment")
print(f"  - Blacklist (commodities): {sum(len(v) for v in BLACKLIST_PATTERNS.values())} patrones")
print(f"  - Blacklist (horizonte t1): {len(HORIZON_BLACKLIST['t1'])} patrones anti-leakage")
print(f"  - Max outlier features: {MAX_OUTLIER_FEATURES}")

✓ Configuración de correcciones cargada:
  - Whitelist: 10 features de sentiment
  - Blacklist (commodities): 12 patrones
  - Blacklist (horizonte t1): 29 patrones anti-leakage
  - Max outlier features: 10


## 1. Carga del Dataset COMPLETO (3236 features)

In [56]:
# Cargar dataset COMPLETO
input_file = PROCESSED_DIR / 'features_final_modeling.csv'

print(f"Cargando {input_file.name}...")
df = pd.read_csv(input_file, parse_dates=['date'])

print(f"\n{'='*80}")
print(f"DATASET COMPLETO CARGADO")
print(f"{'='*80}")
print(f"  Dimensiones: {df.shape[0]:,} filas × {df.shape[1]:,} columnas")
print(f"  Período: {df['date'].min().date()} a {df['date'].max().date()}")
print(f"  Memoria: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# Verificar precios spot
for c in TARGET_COMMODITIES:
    status = "✓" if c in df.columns else "✗"
    print(f"  {c} spot: {status}")

Cargando features_final_modeling.csv...

DATASET COMPLETO CARGADO
  Dimensiones: 20,173 filas × 3,247 columnas
  Período: 2000-01-03 a 2025-11-10
  Memoria: 0.53 GB
  Corn spot: ✓
  Soybeans spot: ✓
  Wheat spot: ✓

DATASET COMPLETO CARGADO
  Dimensiones: 20,173 filas × 3,247 columnas
  Período: 2000-01-03 a 2025-11-10
  Memoria: 0.53 GB
  Corn spot: ✓
  Soybeans spot: ✓
  Wheat spot: ✓


## 2. Creación de Targets de Dirección Multi-Horizonte

In [57]:
# Crear targets binarios de dirección
# Direction = 1 si P_{t+h} > P_t, else 0

targets_dict = {}

for commodity in TARGET_COMMODITIES:
    targets_dict[commodity] = {}
    
    for horizon_name, horizon_days in HORIZONS.items():
        future_price = df[commodity].shift(-horizon_days)
        current_price = df[commodity]
        
        target_col = f'{commodity}_direction_{horizon_name}'
        df[target_col] = (future_price > current_price).astype(float)
        df.loc[df.index[-horizon_days:], target_col] = np.nan
        
        targets_dict[commodity][horizon_name] = target_col

print("="*80)
print("TARGETS DE CLASIFICACIÓN CREADOS")
print("="*80)

# Análisis de balance
for commodity in TARGET_COMMODITIES:
    print(f"\n{commodity}:")
    for horizon_name, target_col in targets_dict[commodity].items():
        valid = df[target_col].notna()
        up_pct = df.loc[valid, target_col].mean() * 100
        n = valid.sum()
        balance = '✓' if 45 <= up_pct <= 55 else '⚠️'
        print(f"  {horizon_name} ({HORIZONS[horizon_name]:2d}d): Sube {up_pct:5.1f}% | n={n:,} | {balance}")

TARGETS DE CLASIFICACIÓN CREADOS

Corn:
  t1 ( 1d): Sube  15.8% | n=20,172 | ⚠️
  t5 ( 5d): Sube  48.4% | n=20,168 | ✓
  t21 (21d): Sube  49.8% | n=20,152 | ✓

Soybeans:
  t1 ( 1d): Sube  16.7% | n=20,172 | ⚠️
  t5 ( 5d): Sube  50.0% | n=20,168 | ✓
  t21 (21d): Sube  50.9% | n=20,152 | ✓

Wheat:
  t1 ( 1d): Sube  15.7% | n=20,172 | ⚠️
  t5 ( 5d): Sube  46.9% | n=20,168 | ✓
  t21 (21d): Sube  47.0% | n=20,152 | ✓


## 3. Identificación de Features vs Metadata

In [58]:
# Columnas que NO son features
non_feature_cols = ['date']

# Targets creados
target_cols_created = [f'{c}_direction_{h}' for c in TARGET_COMMODITIES for h in HORIZONS.keys()]
non_feature_cols.extend(target_cols_created)

# Metadata temporal
temporal_metadata = ['year', 'month', 'quarter', 'day_of_week', 'day_of_year', 
                     'week_of_year', 'is_month_end', 'is_quarter_end', 'is_year_end',
                     'days_since_year_start']
non_feature_cols.extend([c for c in temporal_metadata if c in df.columns])

# Identificar features
feature_cols = [c for c in df.columns if c not in non_feature_cols]

# Verificar que no hay targets en features
problematic = [f for f in feature_cols if 'direction' in f or 'target' in f]
if problematic:
    print(f"⚠️ Removiendo columnas problemáticas: {problematic}")
    feature_cols = [f for f in feature_cols if f not in problematic]

print(f"Total columnas: {len(df.columns)}")
print(f"Columnas excluidas: {len(non_feature_cols)}")
print(f"Feature columns: {len(feature_cols)}")

Total columnas: 3256
Columnas excluidas: 20
Feature columns: 3236


## 4. Limpieza y Train/Test Split

In [59]:
# Eliminar filas sin targets (por shift)
max_horizon = max(HORIZONS.values())
df_clean = df.iloc[:-max_horizon].copy()

# Filtrar solo columnas numéricas de las features
numeric_feature_cols = [c for c in feature_cols if df_clean[c].dtype in ['int64', 'float64', 'int32', 'float32']]
non_numeric = set(feature_cols) - set(numeric_feature_cols)
if len(non_numeric) > 0:
    print(f"⚠️ Columnas no numéricas excluidas: {len(non_numeric)}")
    print(f"   Ejemplos: {list(non_numeric)[:5]}")

feature_cols = numeric_feature_cols
print(f"Features numéricas: {len(feature_cols)}")

# Limpiar NaN/Inf en features
X = df_clean[feature_cols].copy()
X.replace([np.inf, -np.inf], np.nan, inplace=True)

nan_count = X.isnull().sum().sum()
print(f"NaN/Inf en features: {nan_count:,}")

if nan_count > 0:
    X = X.fillna(X.median())
    print("✓ Imputados con mediana")

df_clean[feature_cols] = X

# Split temporal
SPLIT_DATE = '2023-01-01'
train_df = df_clean[df_clean['date'] < SPLIT_DATE].copy()
test_df = df_clean[df_clean['date'] >= SPLIT_DATE].copy()

print(f"\n{'='*80}")
print(f"TRAIN/TEST SPLIT")
print(f"{'='*80}")
print(f"Train: {len(train_df):,} obs ({len(train_df)/len(df_clean)*100:.1f}%)")
print(f"Test:  {len(test_df):,} obs ({len(test_df)/len(df_clean)*100:.1f}%)")

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

del X
gc.collect()

⚠️ Columnas no numéricas excluidas: 2
   Ejemplos: ['commodity', 'format']
Features numéricas: 3234
NaN/Inf en features: 0
NaN/Inf en features: 0

TRAIN/TEST SPLIT
Train: 17,959 obs (89.1%)
Test:  2,193 obs (10.9%)

TRAIN/TEST SPLIT
Train: 17,959 obs (89.1%)
Test:  2,193 obs (10.9%)


0

## 5. ETAPA 1: Variance Threshold

Eliminar features casi constantes (varianza < 0.01 después de estandarizar)

In [60]:
# Estandarizar para calcular varianza comparable
scaler_var = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler_var.fit_transform(X_train),
    columns=feature_cols,
    index=X_train.index
)

VARIANCE_THRESHOLD = 0.01

selector_var = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
selector_var.fit(X_train_scaled)

mask_var = selector_var.get_support()
features_after_var = [f for f, m in zip(feature_cols, mask_var) if m]

print(f"Threshold: {VARIANCE_THRESHOLD}")
print(f"Features iniciales: {len(feature_cols):,}")
print(f"Features removidas: {len(feature_cols) - len(features_after_var):,}")
print(f"Features restantes: {len(features_after_var):,}")

current_features = features_after_var.copy()
del X_train_scaled
gc.collect()

Threshold: 0.01
Features iniciales: 3,234
Features removidas: 178
Features restantes: 3,056


0

## 6. ETAPA 2: High Correlation Filter

Eliminar features redundantes (|r| > 0.95 entre sí)

In [61]:
CORR_THRESHOLD = 0.95

print(f"Calculando matriz de correlación para {len(current_features):,} features...")
X_current = X_train[current_features]

corr_matrix = X_current.corr().abs()

# Triángulo superior
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Features a eliminar
to_drop_corr = set()
for col in upper_triangle.columns:
    high_corr_cols = upper_triangle.index[upper_triangle[col] > CORR_THRESHOLD].tolist()
    to_drop_corr.update(high_corr_cols)

features_after_corr = [f for f in current_features if f not in to_drop_corr]

print(f"\nThreshold: |r| > {CORR_THRESHOLD}")
print(f"Features antes: {len(current_features):,}")
print(f"Features removidas: {len(to_drop_corr):,}")
print(f"Features restantes: {len(features_after_corr):,}")

current_features = features_after_corr.copy()
del corr_matrix, upper_triangle
gc.collect()

Calculando matriz de correlación para 3,056 features...

Threshold: |r| > 0.95
Features antes: 3,056
Features removidas: 1,262
Features restantes: 1,794

Threshold: |r| > 0.95
Features antes: 3,056
Features removidas: 1,262
Features restantes: 1,794


0

## 7. ETAPA 3: Mutual Information Ranking

Ranking de features por información mutua con cada target. Top 150 por commodity-horizonte.

In [62]:
MI_TOP_K = 150

mi_results = {}

for commodity in tqdm(TARGET_COMMODITIES, desc="MI por commodity"):
    mi_results[commodity] = {}
    
    for horizon_name in HORIZONS.keys():
        target_col = f'{commodity}_direction_{horizon_name}'
        
        # Datos válidos
        y_train = train_df[target_col].dropna()
        X_mi = X_train.loc[y_train.index, current_features]
        
        # Calcular MI
        mi_scores = mutual_info_classif(
            X_mi, y_train, 
            discrete_features=False,
            random_state=RANDOM_STATE,
            n_neighbors=5
        )
        
        # Ranking
        mi_ranking = pd.Series(mi_scores, index=current_features).sort_values(ascending=False)
        
        # =====================================================================
        # CORRECCIÓN 1: Limitar features is_outlier a MAX_OUTLIER_FEATURES
        # =====================================================================
        outlier_features = [f for f in mi_ranking.index if 'is_outlier' in f]
        outlier_mi = mi_ranking[outlier_features].sort_values(ascending=False)
        outliers_to_keep = set(outlier_mi.head(MAX_OUTLIER_FEATURES).index)
        outliers_to_remove = set(outlier_mi.index) - outliers_to_keep
        
        # Remover outliers excedentes del ranking
        mi_ranking_filtered = mi_ranking.drop(labels=list(outliers_to_remove), errors='ignore')
        
        # =====================================================================
        # CORRECCIÓN 2: Agregar whitelist features que pasaron variance/corr
        # =====================================================================
        whitelist_in_current = [f for f in WHITELIST_FEATURES if f in current_features]
        
        # Top features de MI (sin whitelist para evitar duplicados)
        top_from_mi = mi_ranking_filtered.head(MI_TOP_K).index.tolist()
        top_from_mi = [f for f in top_from_mi if f not in whitelist_in_current]
        
        # Combinar: whitelist primero, luego top MI hasta MI_TOP_K
        top_features = whitelist_in_current + top_from_mi[:MI_TOP_K - len(whitelist_in_current)]
        
        mi_results[commodity][horizon_name] = {
            'ranking': mi_ranking,
            'top_features': top_features,
            'top_scores': mi_ranking.head(10).to_dict(),
            'whitelist_included': whitelist_in_current,
            'outliers_removed': len(outliers_to_remove)
        }

# Resumen de correcciones
print(f"\n✓ Mutual Information calculada con correcciones")
print(f"  Top K: {MI_TOP_K} por commodity-horizonte")
print(f"  Whitelist forzada: {len(WHITELIST_FEATURES)} features de sentiment")
print(f"  Max outlier features: {MAX_OUTLIER_FEATURES}")

# Verificar que sentiment está incluido
for commodity in TARGET_COMMODITIES:
    n_sentiment = len(mi_results[commodity]['t5']['whitelist_included'])
    n_outliers_removed = mi_results[commodity]['t5']['outliers_removed']
    print(f"  {commodity} (t5): {n_sentiment} sentiment incluidas, {n_outliers_removed} outliers removidos")

MI por commodity:   0%|          | 0/3 [00:00<?, ?it/s]


✓ Mutual Information calculada con correcciones
  Top K: 150 por commodity-horizonte
  Whitelist forzada: 10 features de sentiment
  Max outlier features: 10
  Corn (t5): 9 sentiment incluidas, 144 outliers removidos
  Soybeans (t5): 9 sentiment incluidas, 144 outliers removidos
  Wheat (t5): 9 sentiment incluidas, 144 outliers removidos


In [63]:
# DEBUG: Verificar archivo fuente
print(f"Archivo cargado: {input_file}")
print(f"Columnas totales en df: {len(df.columns)}")

# Buscar cualquier columna con 'tone' o 'article'
tone_cols = [c for c in df.columns if 'tone' in c.lower()]
article_cols = [c for c in df.columns if 'article' in c.lower()]
print(f"\nColumnas con 'tone': {len(tone_cols)}")
print(f"Columnas con 'article': {len(article_cols)}")

if tone_cols:
    print(f"Ejemplos: {tone_cols[:5]}")
else:
    print("¡NO hay columnas de sentiment en el dataset!")
    print("El notebook 2.7 debe re-ejecutarse para generar features_final_modeling.csv con sentiment")

Archivo cargado: C:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed\features_final_modeling.csv
Columnas totales en df: 3256

Columnas con 'tone': 8
Columnas con 'article': 2
Ejemplos: ['tone_mean', 'tone_std', 'tone_ma7', 'tone_ma30', 'tone_volatility_7d']


In [64]:
# Mostrar top features por MI (horizonte t5)
print("TOP 10 FEATURES POR MUTUAL INFORMATION (horizonte t5):")
print("-" * 60)

for commodity in TARGET_COMMODITIES:
    print(f"\n{commodity}:")
    for feat, score in mi_results[commodity]['t5']['top_scores'].items():
        print(f"  {score:.4f}  {feat}")

TOP 10 FEATURES POR MUTUAL INFORMATION (horizonte t5):
------------------------------------------------------------

Corn:
  0.2007  Gold_volume_bb_lower90
  0.1973  Silver_volume_bb_lower90
  0.1886  Gold_volume_vol_ratio_30_90
  0.1831  Gold_volume_bb_lower30
  0.1791  Soybean_Oil_bb_upper90
  0.1780  Precip_Global_Grain_bb_upper30
  0.1774  Treasury_2Y_bb_lower90
  0.1773  Wheat_volume_bb_upper90
  0.1770  Platinum_volume_bb_lower90
  0.1770  Soybean_Oil_volume_bb_upper30

Soybeans:
  0.2038  Gold_volume_bb_lower90
  0.2035  Silver_volume_bb_lower90
  0.1997  Gold_volume_bb_lower30
  0.1988  Gold_volume_vol_ratio_30_90
  0.1937  Silver_volume_bb_lower30
  0.1900  Copper_volume_bb_lower90
  0.1886  Heating_Oil_volume_bb_upper7
  0.1860  Wheat_volume_bb_upper90
  0.1859  Platinum_volume_bb_upper90
  0.1848  Silver_volume_vol_ratio_30_90

Wheat:
  0.1908  Gold_volume_bb_lower90
  0.1882  Silver_volume_bb_lower90
  0.1818  Gold_volume_bb_lower30
  0.1797  Silver_volume_vol_ratio_30_90
 

## 8. ETAPA 4: LASSO-Based Selection

En esta etapa final utilizamos Logistic Regression con penalización L1 (LASSO) para realizar selección embebida (embedded selection). LASSO elimina automáticamente features irrelevantes al shrinkear sus coeficientes exactamente a cero.

Ventajas de LASSO para feature selection:
- Selección automática de variables relevantes
- Maneja multicolinealidad residual
- Produce modelos más interpretables
- El parámetro C controla la intensidad de la regularización

In [65]:
# LASSO-based selection usando las top MI features
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Parámetro C para LASSO (inverso de lambda)
# C más bajo = más regularización = menos features
LASSO_C = 0.1

lasso_results = {}

for commodity in TARGET_COMMODITIES:
    print(f"\n{'='*60}")
    print(f"LASSO SELECTION: {commodity}")
    print('='*60)
    
    lasso_results[commodity] = {}
    
    for horizon in HORIZONS:
        target_col = f'{commodity}_direction_{horizon}'
        
        # Features después de MI ranking
        mi_features = mi_results[commodity][horizon]['top_features']
        
        # =====================================================================
        # CORRECCIÓN 3: Aplicar BLACKLIST - remover correlaciones espurias
        # =====================================================================
        blacklist_for_commodity = BLACKLIST_PATTERNS.get(commodity, [])
        mi_features_filtered = [
            f for f in mi_features 
            if not any(bl_pattern in f for bl_pattern in blacklist_for_commodity)
        ]
        n_blacklisted = len(mi_features) - len(mi_features_filtered)
        
        # =====================================================================
        # CORRECCIÓN 3b: Aplicar HORIZON_BLACKLIST - evitar data leakage
        # =====================================================================
        horizon_blacklist = HORIZON_BLACKLIST.get(horizon, [])
        if horizon_blacklist:
            n_before_horizon = len(mi_features_filtered)
            mi_features_filtered = [
                f for f in mi_features_filtered 
                if not any(bl_pattern in f for bl_pattern in horizon_blacklist)
            ]
            n_horizon_removed = n_before_horizon - len(mi_features_filtered)
            if n_horizon_removed > 0:
                print(f"    ⚠️ LEAKAGE REMOVED: {n_horizon_removed} features (CFTC/gov delay)")
        
        # Preparar datos (usar X_train con las features de MI filtradas)
        X_train_h = X_train[mi_features_filtered].copy()
        y_train_h = train_df[target_col]
        
        # Manejar NaN en target
        valid_idx = y_train_h.notna()
        X_train_valid = X_train_h[valid_idx]
        y_train_valid = y_train_h[valid_idx]
        
        # Escalar features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_train_valid)
        
        # Aplicar LASSO
        lasso = LogisticRegression(
            penalty='l1',
            C=LASSO_C,
            solver='saga',
            max_iter=2000,
            class_weight='balanced',
            random_state=42
        )
        lasso.fit(X_scaled, y_train_valid)
        
        # Features con coeficientes no-cero
        non_zero_mask = lasso.coef_[0] != 0
        selected_features = [mi_features_filtered[i] for i in range(len(mi_features_filtered)) if non_zero_mask[i]]
        non_zero_coefs = lasso.coef_[0][non_zero_mask]
        
        # =====================================================================
        # CORRECCIÓN 4: Proteger WHITELIST - agregar aunque LASSO las elimine
        # =====================================================================
        whitelist_in_mi = [f for f in WHITELIST_FEATURES if f in mi_features_filtered]
        whitelist_not_selected = [f for f in whitelist_in_mi if f not in selected_features]
        
        # Agregar whitelist que LASSO eliminó (con coef 0.01 para tracking)
        for wl_feat in whitelist_not_selected:
            selected_features.append(wl_feat)
            non_zero_coefs = np.append(non_zero_coefs, 0.01)  # Coef pequeño para indicar forzado
        
        # Ordenar por magnitud absoluta del coeficiente
        sorted_idx = np.argsort(np.abs(non_zero_coefs))[::-1]
        selected_features_sorted = [selected_features[i] for i in sorted_idx]
        coefs_sorted = non_zero_coefs[sorted_idx]
        
        lasso_results[commodity][horizon] = {
            'selected_features': selected_features_sorted,
            'n_selected': len(selected_features_sorted),
            'coefficients': dict(zip(selected_features_sorted, coefs_sorted)),
            'scaler': scaler,
            'model': lasso,
            'n_blacklisted': n_blacklisted,
            'n_whitelist_forced': len(whitelist_not_selected)
        }
        
        print(f"\n  {horizon}: {len(selected_features_sorted)} features seleccionadas")
        print(f"       (de {len(mi_features)} MI → {len(mi_features_filtered)} post-blacklist)")
        if n_blacklisted > 0:
            print(f"       Blacklist removió: {n_blacklisted} features")
        if len(whitelist_not_selected) > 0:
            print(f"       Whitelist forzó: {len(whitelist_not_selected)} features")
        if len(selected_features_sorted) > 0:
            # Mostrar cuántas son de sentiment
            n_sentiment = len([f for f in selected_features_sorted if f in WHITELIST_FEATURES])
            print(f"       Top 5: {selected_features_sorted[:5]}")
            print(f"       Sentiment features: {n_sentiment}")


LASSO SELECTION: Corn
    ⚠️ LEAKAGE REMOVED: 75 features (CFTC/gov delay)

  t1: 63 features seleccionadas
       (de 150 MI → 73 post-blacklist)
       Blacklist removió: 2 features
       Whitelist forzó: 1 features
       Top 5: ['open_interest', 'tone_ma30', 'tone_std', 'tone_momentum_7d', 'tone_volatility_30d']
       Sentiment features: 9

  t1: 63 features seleccionadas
       (de 150 MI → 73 post-blacklist)
       Blacklist removió: 2 features
       Whitelist forzó: 1 features
       Top 5: ['open_interest', 'tone_ma30', 'tone_std', 'tone_momentum_7d', 'tone_volatility_30d']
       Sentiment features: 9

  t5: 110 features seleccionadas
       (de 150 MI → 138 post-blacklist)
       Blacklist removió: 12 features
       Whitelist forzó: 1 features
       Top 5: ['Gold_bb_lower90', 'Corn_bb_upper90', 'Treasury_10Y_bb_lower90', 'Crude_Oil_volume_bb_upper90', 'Coffee_volume_bb_upper90']
       Sentiment features: 9

  t5: 110 features seleccionadas
       (de 150 MI → 138 post-

## 9. Resumen del Pipeline de Selección

Consolidamos los resultados del pipeline completo de 4 etapas para cada commodity y horizonte temporal.

In [66]:
# Crear DataFrame resumen del pipeline
# Usar variables correctas del pipeline

n_original = 3224  # feature_cols después del filtro numérico
n_after_var = len(features_after_var)
n_after_corr = len(features_after_corr)

summary_data = []

for commodity in TARGET_COMMODITIES:
    for horizon in HORIZONS:
        n_mi = len(mi_results[commodity][horizon]['top_features'])
        n_lasso = lasso_results[commodity][horizon]['n_selected']
        
        summary_data.append({
            'Commodity': commodity,
            'Horizon': horizon,
            'Original': n_original,
            'Post-Variance': n_after_var,
            'Post-Correlation': n_after_corr,
            'Post-MI': n_mi,
            'Post-LASSO': n_lasso,
            'Reduction_%': round(100 * (1 - n_lasso / n_original), 1)
        })

summary_df = pd.DataFrame(summary_data)
print("RESUMEN DEL PIPELINE DE FEATURE SELECTION")
print("="*80)
display(summary_df)

RESUMEN DEL PIPELINE DE FEATURE SELECTION


,Commodity,Horizon,Original,Post-Variance,Post-Correlation,Post-MI,Post-LASSO,Reduction_%
0,Corn,t1,3224,3056,1794,150,63,98.0
1,Corn,t5,3224,3056,1794,150,110,96.6
2,Corn,t21,3224,3056,1794,150,108,96.7
3,Soybeans,t1,3224,3056,1794,150,54,98.3
4,Soybeans,t5,3224,3056,1794,150,110,96.6
5,Soybeans,t21,3224,3056,1794,150,106,96.7
6,Wheat,t1,3224,3056,1794,150,61,98.1
7,Wheat,t5,3224,3056,1794,150,112,96.5
8,Wheat,t21,3224,3056,1794,150,108,96.7


## 10. Análisis de Features Comunes entre Commodities

Identificamos features que son predictivas para múltiples commodities, lo cual sugiere factores de mercado sistemáticos.

In [67]:
# Analizar overlap de features entre commodities (horizonte t5)
horizon_analysis = 't5'

feature_sets = {}
for commodity in TARGET_COMMODITIES:
    feature_sets[commodity] = set(lasso_results[commodity][horizon_analysis]['selected_features'])

# Features comunes
common_all = feature_sets['Corn'] & feature_sets['Soybeans'] & feature_sets['Wheat']
common_corn_soy = feature_sets['Corn'] & feature_sets['Soybeans'] - common_all
common_corn_wheat = feature_sets['Corn'] & feature_sets['Wheat'] - common_all
common_soy_wheat = feature_sets['Soybeans'] & feature_sets['Wheat'] - common_all

print(f"ANÁLISIS DE OVERLAP (Horizonte: {horizon_analysis})")
print("="*60)
print(f"\nFeatures comunes a los 3 commodities: {len(common_all)}")
if len(common_all) > 0:
    print(f"  {list(common_all)[:10]}...")
    
print(f"\nFeatures Corn-Soybeans (excl. Wheat): {len(common_corn_soy)}")
print(f"Features Corn-Wheat (excl. Soybeans): {len(common_corn_wheat)}")
print(f"Features Soybeans-Wheat (excl. Corn): {len(common_soy_wheat)}")

# Features únicas
for commodity in TARGET_COMMODITIES:
    unique = feature_sets[commodity] - feature_sets[(set(TARGET_COMMODITIES) - {commodity}).pop()] - feature_sets[list(set(TARGET_COMMODITIES) - {commodity})[1 if len(set(TARGET_COMMODITIES) - {commodity}) > 1 else 0]]
    print(f"\nFeatures únicas de {commodity}: {len(unique)}")

ANÁLISIS DE OVERLAP (Horizonte: t5)

Features comunes a los 3 commodities: 25
  ['Natural_Gas_volume_vol_ratio_30_90', 'Copper_bb_lower90', 'RBOB_Gasoline_bb_upper90', 'Soybeans_volume_bb_lower90', 'tone_volatility_7d', 'tone_percentile_30d', 'tone_std', 'Crude_Oil_volume_bb_upper90', 'Sugar_volume_bb_upper90', 'article_count']...

Features Corn-Soybeans (excl. Wheat): 19
Features Corn-Wheat (excl. Soybeans): 29
Features Soybeans-Wheat (excl. Corn): 21

Features únicas de Corn: 37

Features únicas de Soybeans: 45

Features únicas de Wheat: 37


## 11. Guardar Features Seleccionadas

Exportamos las features seleccionadas y el dataset reducido para usar en los modelos de clasificación.

In [68]:
# Crear directorio de salida (usar PROCESSED_DIR de config)
output_dir = PROCESSED_DIR / 'final_modeling'
output_dir.mkdir(parents=True, exist_ok=True)

# Guardar diccionario de features seleccionadas

features_dict = {}
for commodity in TARGET_COMMODITIES:
    features_dict[commodity] = {}
    for horizon in HORIZONS:
        features_dict[commodity][horizon] = lasso_results[commodity][horizon]['selected_features']

# Guardar como JSON
with open(output_dir / 'selected_features.json', 'w') as f:
    json.dump(features_dict, f, indent=2)

print(f"Features guardadas en: {output_dir / 'selected_features.json'}")

Features guardadas en: C:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed\final_modeling\selected_features.json


In [69]:
# Crear y guardar datasets reducidos para cada commodity
for commodity in TARGET_COMMODITIES:
    # Unión de features de todos los horizontes para este commodity
    all_features = set()
    for horizon in HORIZONS:
        all_features.update(lasso_results[commodity][horizon]['selected_features'])
    
    # Crear dataset con targets y features seleccionadas
    cols_to_keep = ['date'] + list(all_features)
    
    # Agregar targets
    for horizon in HORIZONS:
        target_col = f'{commodity}_direction_{horizon}'
        if target_col in df.columns:
            cols_to_keep.append(target_col)
    
    # Filtrar columnas existentes
    cols_existing = [c for c in cols_to_keep if c in df.columns]
    df_subset = df[cols_existing].copy()
    
    # Guardar
    output_file = output_dir / f'features_{commodity.lower()}_optimized.csv'
    df_subset.to_csv(output_file, index=False)
    
    print(f"{commodity}: {len(all_features)} features únicas guardadas en {output_file.name}")

print(f"\n✓ Datasets guardados en: {output_dir}")

Corn: 219 features únicas guardadas en features_corn_optimized.csv
Soybeans: 215 features únicas guardadas en features_soybeans_optimized.csv
Soybeans: 215 features únicas guardadas en features_soybeans_optimized.csv
Wheat: 218 features únicas guardadas en features_wheat_optimized.csv

✓ Datasets guardados en: C:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed\final_modeling
Wheat: 218 features únicas guardadas en features_wheat_optimized.csv

✓ Datasets guardados en: C:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed\final_modeling


In [70]:
# Guardar también información del pipeline para reproducibilidad
pipeline_info = {
    'config': {
        'VARIANCE_THRESHOLD': VARIANCE_THRESHOLD,
        'CORR_THRESHOLD': CORR_THRESHOLD,
        'MI_TOP_K': MI_TOP_K,
        'LASSO_C': LASSO_C,
        'SPLIT_DATE': SPLIT_DATE,
        'HORIZONS': HORIZONS,
        'TARGET_COMMODITIES': TARGET_COMMODITIES
    },
    'pipeline_summary': {
        'original_features': 3224,
        'after_variance': len(features_after_var),
        'after_correlation': len(features_after_corr)
    }
}

with open(output_dir / 'pipeline_config.json', 'w') as f:
    json.dump(pipeline_info, f, indent=2, default=str)

print(f"Configuración del pipeline guardada en: {output_dir / 'pipeline_config.json'}")

Configuración del pipeline guardada en: C:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed\final_modeling\pipeline_config.json


## 12. Conclusiones

### Pipeline de 4 Etapas Completado

1. **Variance Threshold**: Eliminación de features con varianza casi nula
2. **Correlation Filter**: Eliminación de features altamente correlacionadas (>0.95)
3. **Mutual Information**: Ranking y selección de top-K features más informativas
4. **LASSO Selection**: Selección embebida final con regularización L1

### Próximos Pasos

Los datasets optimizados están listos para entrenar modelos:
- `4.2-logit-lasso-optimized.ipynb`: Logistic Regression con L1/L2
- Futuros: Random Forest, XGBoost, LSTM, etc.

Las features seleccionadas se encuentran en:
- `data/processed/final_modeling/selected_features.json`
- `data/processed/final_modeling/features_{commodity}_optimized.csv`

In [71]:
# =============================================================================
# DIAGNÓSTICO DE DATA LEAKAGE - Análisis de Features Sospechosas
# =============================================================================
print("="*80)
print("🚨 DIAGNÓSTICO DE DATA LEAKAGE")
print("="*80)

# Features que probablemente tienen lag de publicación
LEAKAGE_SUSPECTS = {
    # CFTC Commitments of Traders - publicados viernes con 3 días de lag
    'cftc_patterns': ['_long', '_short', '_net', 'swap_', 'managed_', 'producer_', 'other_'],
    
    # Government stocks/inventory - reportados con delay semanal/mensual
    'gov_stocks': ['gov_stocks', 'inventory', 'stocks_'],
    
    # USDA reports - mensuales con fecha específica
    'usda': ['usda_', 'wasde_', 'crop_progress'],
}

print("\n📊 ANÁLISIS POR HORIZONTE:")
print("-"*80)

for horizon in ['t1', 't5', 't21']:
    print(f"\n=== Horizonte: {horizon} ===")
    
    for commodity in TARGET_COMMODITIES:
        features = lasso_results[commodity][horizon]['selected_features']
        
        # Buscar patterns sospechosos
        cftc_features = [f for f in features if any(p in f.lower() for p in LEAKAGE_SUSPECTS['cftc_patterns'])]
        gov_features = [f for f in features if any(p in f.lower() for p in LEAKAGE_SUSPECTS['gov_stocks'])]
        
        print(f"\n  {commodity}:")
        print(f"    Total features: {len(features)}")
        
        if cftc_features:
            print(f"    ⚠️ CFTC/COT features ({len(cftc_features)}): {cftc_features[:5]}...")
            
        if gov_features:
            print(f"    ⚠️ Government stocks ({len(gov_features)}): {gov_features}")

print("\n" + "="*80)
print("💡 RECOMENDACIONES:")
print("="*80)
print("""
1. t1 (1 día): AUC=0.88 es IMPOSIBLE → Data leakage confirmado
   - CFTC COT tiene 3 días de lag → NO usar para t1
   - Solución: Agregar CFTC a blacklist para t1

2. t5 (5 días): AUC=0.53 es REALISTA ✓
   - CFTC COT puede ser válido (5 días > 3 días lag)
   - Mantener configuración actual

3. t21 (21 días): AUC=0.47-0.62 REALISTA ✓
   - Todos los datos tienen tiempo de incorporarse
   - Configuración OK

ACCIÓN RECOMENDADA:
- Crear blacklist ESPECÍFICA por horizonte
- t1: Excluir TODAS las features CFTC y gov_stocks
""")

🚨 DIAGNÓSTICO DE DATA LEAKAGE

📊 ANÁLISIS POR HORIZONTE:
--------------------------------------------------------------------------------

=== Horizonte: t1 ===

  Corn:
    Total features: 63

  Soybeans:
    Total features: 54

  Wheat:
    Total features: 61

=== Horizonte: t5 ===

  Corn:
    Total features: 110

  Soybeans:
    Total features: 110

  Wheat:
    Total features: 112

=== Horizonte: t21 ===

  Corn:
    Total features: 108

  Soybeans:
    Total features: 106

  Wheat:
    Total features: 108

💡 RECOMENDACIONES:

1. t1 (1 día): AUC=0.88 es IMPOSIBLE → Data leakage confirmado
   - CFTC COT tiene 3 días de lag → NO usar para t1
   - Solución: Agregar CFTC a blacklist para t1

2. t5 (5 días): AUC=0.53 es REALISTA ✓
   - CFTC COT puede ser válido (5 días > 3 días lag)
   - Mantener configuración actual

3. t21 (21 días): AUC=0.47-0.62 REALISTA ✓
   - Todos los datos tienen tiempo de incorporarse
   - Configuración OK

ACCIÓN RECOMENDADA:
- Crear blacklist ESPECÍFICA por 